# High-Performance Dataset Cleaner v3 — Dask + cuDF (91.5 GB / RTX 3070 8 GB)

## v2 → v3 changes

| Problem in v2 | Fix in v3 |
|---|---|
| Plain cuDF loads whole file → VRAM spill on large files | **Dask-cuDF** partitions lazily — each partition ≤ 1 GB in VRAM |
| ProcessPoolExecutor + manual chunk loop | Dask task graph handles scheduling automatically |
| AsyncWriter queue — still CPU-bound for writing | `ddf.to_parquet()` — parallel Parquet write, GPU-native |
| Single output file → full scan every training run | **Partitioned Parquet** by `source_file` — load only what you need |
| All columns loaded (91.5 GB) | `KEEP_COLUMNS` whitelist — skip irrelevant columns at read time |
| Schema scan hits every file header serially | Parallel schema scan with `ThreadPoolExecutor` |
| Medians sampled per-file sequentially | Dask `ddf.median_approximate()` — distributed approximation |


In [25]:
%pip install dask[dataframe]

Note: you may need to restart the kernel to use updated packages.


In [26]:
%pip install "dask[distributed]" --upgrade

Note: you may need to restart the kernel to use updated packages.


In [27]:
# -- Cell 2: Dask cluster setup (stable for 8 GB GPU + WSL2) -----------------
import os
import dask
import psutil

# Decide GPU/CPU mode once so later cells can reuse USE_GPU and cp
try:
    import cupy as cp
except Exception:
    cp = None

try:
    from dask_cuda import LocalCUDACluster
    _HAS_DASK_CUDA = True
except Exception:
    LocalCUDACluster = None
    _HAS_DASK_CUDA = False

USE_GPU = (cp is not None) and _HAS_DASK_CUDA

# Conservative defaults to prevent nanny restart loops on 8 GB VRAM
DASK_SCRATCH = '/mnt/e/thesis/dask-scratch'
os.makedirs(DASK_SCRATCH, exist_ok=True)

total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
host_mem_limit_gb = max(8, int(total_ram_gb * 0.60))

dask.config.set({
    'distributed.worker.memory.target': 0.80,
    'distributed.worker.memory.spill': 0.90,
    'distributed.worker.memory.pause': 0.95,
    'distributed.worker.memory.terminate': 0.98,
    'distributed.scheduler.allowed-failures': 1,
    'distributed.comm.timeouts.connect': '60s',
    'distributed.comm.timeouts.tcp': '120s',
})

if USE_GPU:
    cluster = LocalCUDACluster(
        n_workers=1,
        threads_per_worker=1,
        CUDA_VISIBLE_DEVICES='0',
        protocol='tcp',
        local_directory=DASK_SCRATCH,
        device_memory_limit='5.5GB',
        memory_limit=f'{host_mem_limit_gb}GB',
        rmm_pool_size='4GB',
        rmm_async=True,
        enable_cudf_spill=True,
        jit_unspill=False,
    )
else:
    from dask.distributed import LocalCluster
    cluster = LocalCluster(
        n_workers=min(2, os.cpu_count() or 2),
        threads_per_worker=1,
        memory_limit=f'{host_mem_limit_gb}GB',
        local_directory=DASK_SCRATCH,
    )

from dask.distributed import Client
client = Client(cluster)

print(f'USE_GPU={USE_GPU}')
print(f'host_mem_limit={host_mem_limit_gb}GB')
print('dashboard:', client.dashboard_link)
client

/home/devmrx/miniforge3/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 33863 instead
  warnings.warn(


USE_GPU=False
host_mem_limit=16GB
dashboard: http://127.0.0.1:33863/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:33863/status,
Dashboard: http://127.0.0.1:33863/status,Workers: 2
Total threads: 2,Total memory: 29.80 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33237,Workers: 0
Dashboard: http://127.0.0.1:33863/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:37653,Total threads: 1
Dashboard: http://127.0.0.1:37673/status,Memory: 14.90 GiB
Nanny: tcp://127.0.0.1:41143,


2026-04-05 22:17:54,711 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:39477 (pid=22630) exceeded 98% memory budget. Restarting...
2026-04-05 22:17:56,430 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:39477' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('clean_partition-37eac6c80d26f41c06f30e1d38cd451f', 64), ('reset_index-f20054b57d192f64d747a7511950e867', 9)} (stimulus_id='handle-worker-cleanup-1775402276.4181387')
2026-04-05 22:18:41,500 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:37313 (pid=23043) exceeded 98% memory budget. Restarting...
2026-04-05 22:18:42,255 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:37313' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('clean_partition-37eac6c80d26f41c06f30e1d38cd451f', 98), ('clean_partition-37eac6c80d26f41c06f30e1d38cd451f', 99)} (stimulus_id='handle-worker-cleanup-17754023

In [28]:
from pathlib import Path

# -- Cell 3: config (primary, stability tuned) --------------------------------
DATASET_DIR = Path(r'E:/thesis/dataset')
OUTPUT_DIR = DATASET_DIR / 'dataset_cleaned_parquet'
RUN_DIR = OUTPUT_DIR / 'run_outputs'

# Optional whitelist to reduce memory footprint. Keep None to keep all columns.
KEEP_COLUMNS = None
# KEEP_COLUMNS = ['timestamp', 'source_ip', 'dest_ip', 'protocol', 'label']

# Keep partitions conservative on 8 GB VRAM
PARTITION_SIZE = '256MB'

MEDIAN_ACCURACY = 50
PARQUET_COMPRESSION = 'snappy'
PARQUET_ROW_GROUP_SIZE = 500_000

DELIMITER_CANDIDATES = [',', '\t', '|', ';']
ENCODING_FALLBACKS = ['utf-8', 'latin-1', 'cp1252']
LOW_CAR_MAX_UNIQUE = 1000
LOW_CAR_MAX_RATIO = 0.5
SCHEMA_SCAN_WORKERS = min(8, os.cpu_count() or 4)
MEDIAN_SAMPLE_ROWS = 100_000

EXCLUDE_FILES = {
    'labelled_testing_data.csv',
    'labelled_training_data.csv',
    'labelled_validation_data.csv',
    'dataset_cleaned.csv',
}

FAIL_LOG_PATH = Path('/mnt/e/thesis') / 'dataset_cleaning_failures.jsonl'
INPROGRESS_PATH = Path('/mnt/e/thesis') / 'dataset_cleaning_in_progress.txt'
VALIDATE_ONE_FILE_ONLY = False

if not DATASET_DIR.exists():
    _alt = Path(str(DATASET_DIR).replace('E:/', '/mnt/e/').replace('E:\\', '/mnt/e/'))
    if _alt.exists():
        DATASET_DIR = _alt
        OUTPUT_DIR = DATASET_DIR / 'dataset_cleaned_parquet'
        RUN_DIR = OUTPUT_DIR / 'run_outputs'
    else:
        raise FileNotFoundError(f'Dataset dir not found: {DATASET_DIR}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f'DATASET_DIR={DATASET_DIR}')
print(f'OUTPUT_DIR={OUTPUT_DIR}')
print(f'RUN_DIR={RUN_DIR}')
print(f'FAIL_LOG_PATH={FAIL_LOG_PATH}')
print(f'INPROGRESS_PATH={INPROGRESS_PATH}')

DATASET_DIR=/mnt/e/thesis/dataset
OUTPUT_DIR=/mnt/e/thesis/dataset/dataset_cleaned_parquet
RUN_DIR=/mnt/e/thesis/dataset/dataset_cleaned_parquet/run_outputs
FAIL_LOG_PATH=/mnt/e/thesis/dataset_cleaning_failures.jsonl
INPROGRESS_PATH=/mnt/e/thesis/dataset_cleaning_in_progress.txt


In [29]:
# -- Cell 3b: config mirror (re-run safe, stability tuned) ---------------------
# This cell intentionally mirrors the primary config cell so later executions
# don't accidentally override values with stale settings.

DATASET_DIR = Path(r'E:/thesis/dataset')
OUTPUT_DIR = DATASET_DIR / 'dataset_cleaned_parquet'
RUN_DIR = OUTPUT_DIR / 'run_outputs'

KEEP_COLUMNS = None
PARTITION_SIZE = '256MB'

MEDIAN_ACCURACY = 50
PARQUET_COMPRESSION = 'snappy'
PARQUET_ROW_GROUP_SIZE = 500_000

DELIMITER_CANDIDATES = [',', '\t', '|', ';']
ENCODING_FALLBACKS = ['utf-8', 'latin-1', 'cp1252']
LOW_CAR_MAX_UNIQUE = 1000
LOW_CAR_MAX_RATIO = 0.5
SCHEMA_SCAN_WORKERS = min(8, os.cpu_count() or 4)
MEDIAN_SAMPLE_ROWS = 100_000

EXCLUDE_FILES = {
    'labelled_testing_data.csv',
    'labelled_training_data.csv',
    'labelled_validation_data.csv',
    'dataset_cleaned.csv',
}

FAIL_LOG_PATH = Path('/mnt/e/thesis') / 'dataset_cleaning_failures.jsonl'
INPROGRESS_PATH = Path('/mnt/e/thesis') / 'dataset_cleaning_in_progress.txt'
VALIDATE_ONE_FILE_ONLY = False

if not DATASET_DIR.exists():
    _alt = Path(str(DATASET_DIR).replace('E:/', '/mnt/e/').replace('E:\\', '/mnt/e/'))
    if _alt.exists():
        DATASET_DIR = _alt
        OUTPUT_DIR = DATASET_DIR / 'dataset_cleaned_parquet'
        RUN_DIR = OUTPUT_DIR / 'run_outputs'
    else:
        raise FileNotFoundError(f'Dataset dir not found: {DATASET_DIR}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f'DATASET_DIR={DATASET_DIR}')
print(f'OUTPUT_DIR={OUTPUT_DIR}')
print(f'RUN_DIR={RUN_DIR}')

DATASET_DIR=/mnt/e/thesis/dataset
OUTPUT_DIR=/mnt/e/thesis/dataset/dataset_cleaned_parquet
RUN_DIR=/mnt/e/thesis/dataset/dataset_cleaned_parquet/run_outputs


In [30]:
# -- Cell 4: helpers -------------------------------------------------------------
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm.auto import tqdm

def normalize_col(name: str) -> str:
    name = str(name).strip().lower()
    name = re.sub(r'[^a-z0-9]+', '_', name)
    name = re.sub(r'_+', '_', name).strip('_')
    return name or 'unnamed_column'


def make_unique_columns(columns) -> list:
    seen, out = {}, []
    for col in columns:
        c = normalize_col(col)
        n = seen.get(c, 0)
        seen[c] = n + 1
        out.append(c if n == 0 else f'{c}_{n}')
    return out


def detect_delimiter(path: Path, encoding: str = 'utf-8') -> str:
    try:
        with open(path, 'r', encoding=encoding, errors='replace') as f:
            sample = f.read(65536)
        counts = {d: sample.count(d) for d in DELIMITER_CANDIDATES}
        best = max(counts, key=counts.get)
        return best if counts[best] > 0 else ','
    except Exception:
        return ','


def scan_input_files(root: Path) -> list:
    files = []
    for p in root.rglob('*'):
        if not p.is_file():
            continue
        if p.name in EXCLUDE_FILES:
            continue
        if str(p).startswith(str(OUTPUT_DIR)):
            continue
        if p.suffix.lower() in {'.csv', '.txt'}:
            files.append(p)
    return sorted(files, key=lambda x: x.stat().st_size, reverse=True)


def get_header_columns(path: Path) -> list:
    """Read only the header row -- fast, no GPU needed."""
    for enc in ENCODING_FALLBACKS:
        try:
            sep = detect_delimiter(path, enc)
            df = pd.read_csv(
                path,
                sep=sep,
                encoding=enc,
                nrows=0,
                low_memory=False,
                on_bad_lines='skip',
            )
            return make_unique_columns(df.columns)
        except Exception:
            continue
    return []


def build_global_schema(files: list) -> tuple:
    """Parallel header scan -- ThreadPoolExecutor, no GPU needed."""
    registry, failures = set(), []
    with ThreadPoolExecutor(max_workers=SCHEMA_SCAN_WORKERS) as pool:
        futs = {pool.submit(get_header_columns, f): f for f in files}
        for fut in tqdm(as_completed(futs), total=len(futs), desc='Schema scan', unit='file', leave=False):
            f = futs[fut]
            try:
                cols = fut.result()
                if cols:
                    registry.update(cols)
                else:
                    failures.append((str(f), 'empty header'))
            except Exception as e:
                failures.append((str(f), str(e)))

    registry.add('source_file')
    schema = sorted(registry)
    if KEEP_COLUMNS:
        keep = set(KEEP_COLUMNS) | {'source_file'}
        schema = [c for c in schema if c in keep]
    return schema, failures

In [31]:
# ── Cell 5: pre-compute global medians ───────────────────────────────────────
# Strategy: sample MEDIAN_SAMPLE_ROWS rows from each file on CPU (fast, no
# VRAM used). Compute medians once. Broadcast as a plain dict to Dask workers.
# Workers use the dict for fillna — zero GPU reduction during cleaning.

def sample_medians(files: list, schema_columns: list) -> dict:
    frames = []
    for path in tqdm(files, desc='Sampling medians', unit='file', leave=False):
        for enc in ENCODING_FALLBACKS:
            try:
                sep = detect_delimiter(path, enc)
                cols_to_read = None
                if KEEP_COLUMNS:
                    # Only sample columns we care about
                    hdr = pd.read_csv(path, sep=sep, encoding=enc, nrows=0,
                                      on_bad_lines='skip')
                    available = make_unique_columns(hdr.columns)
                    cols_to_read = [c for c in KEEP_COLUMNS if c in available]
                df = pd.read_csv(
                    path, sep=sep, encoding=enc,
                    nrows=MEDIAN_SAMPLE_ROWS,
                    usecols=cols_to_read,
                    low_memory=False, on_bad_lines='skip',
                )
                df.columns = make_unique_columns(df.columns)
                frames.append(df)
                break
            except Exception:
                continue

    if not frames:
        return {}

    combined = pd.concat(frames, ignore_index=True, sort=False)
    medians  = {}
    for col in schema_columns:
        if col in combined.columns:
            s = pd.to_numeric(combined[col], errors='coerce')
            v = s.median()
            if pd.notna(v):
                medians[col] = float(v)
    del combined, frames
    gc.collect()
    return medians


In [32]:
# -- Cell 7: per-file Dask pipeline (restart-resistant) -------------------------
import ctypes
import gc
import json
import re
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import dask_cudf
except Exception:
    dask_cudf = None


def trim_ram():
    gc.collect()
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass


def is_transient_worker_error(error_text: str) -> bool:
    """Retry only worker/process instability; do not retry deterministic data errors."""
    lower = str(error_text).lower()

    fatal_markers = (
        'mismatched dtypes found',
        'could not convert string to float',
        'integer column has na values',
        'parsererror',
        'tokenizing data',
        'dtype',
        'expected',
        'header parse failed',
        'usecols do not match columns',
        'failed to convert partition to expected pyarrow schema',
    )
    if any(marker in lower for marker in fatal_markers):
        return False

    transient_markers = (
        'worker process died unexpectedly',
        'killedworker',
        'stream is closed',
        'commclosederror',
        'nanny',
        'cuda out of memory',
        'cuda error',
        'out of memory',
        'connection reset',
    )
    return any(marker in lower for marker in transient_markers)


def preflight_file(path: Path):
    """Fast validation to skip incompatible files before GPU tasks are scheduled."""
    if not path.exists():
        return False, f'file not found: {path}'
    if path.stat().st_size == 0:
        return False, 'empty file'

    parse_errors = []
    for enc in ENCODING_FALLBACKS:
        try:
            sep = detect_delimiter(path, enc)
            hdr = pd.read_csv(
                path,
                sep=sep,
                encoding=enc,
                nrows=0,
                low_memory=False,
                on_bad_lines='skip',
            )
            _ = pd.read_csv(
                path,
                sep=sep,
                encoding=enc,
                nrows=5,
                low_memory=False,
                on_bad_lines='skip',
            )
            return True, {'sep': sep, 'enc': enc, 'raw_columns': list(hdr.columns)}
        except Exception as e:
            parse_errors.append(f'{enc}: {e}')

    return False, 'header parse failed: ' + ' | '.join(parse_errors[:3])


def _build_column_maps(raw_columns):
    """Map raw header names to normalized unique names used internally."""
    normalized = make_unique_columns(raw_columns)
    raw_to_norm = {raw: norm for raw, norm in zip(raw_columns, normalized)}
    return raw_to_norm, normalized


def _build_usecols_and_dtype(path: Path, sep: str, enc: str, raw_columns, raw_to_norm):
    """Build read_csv kwargs without generating non-existent usecols names."""
    raw_usecols = None
    if KEEP_COLUMNS:
        keep = set(KEEP_COLUMNS)
        raw_usecols = [c for c in raw_columns if raw_to_norm.get(c) in keep]
        if not raw_usecols:
            raw_usecols = None

    sample = pd.read_csv(
        path,
        sep=sep,
        encoding=enc,
        nrows=4096,
        low_memory=False,
        on_bad_lines='skip',
        usecols=raw_usecols,
    )

    # Keep CSV read stable: numeric -> float32, bool -> bool, others -> str
    dtype_map = {}
    for col, dt in sample.dtypes.items():
        if pd.api.types.is_bool_dtype(dt):
            dtype_map[col] = 'bool'
        elif pd.api.types.is_numeric_dtype(dt):
            dtype_map[col] = 'float32'
        else:
            dtype_map[col] = 'str'

    return raw_usecols, dtype_map


def clean_partition(
    df,
    source_name,
    global_medians,
    schema_columns,
    low_car_max_unique,
    low_car_max_ratio,
    selected_cols,
    string_cols,
    numeric_cols,
    raw_to_norm,
    norm_to_raw,
    use_gpu,
    use_categories=False,
    category_max=120,
    category_ratio=0.05,
    category_exclude_prefixes=('uid', 'community_id', 'datetime', 'history', 'src_ip', 'dest_ip', 'label_technique', 'label_tactic', 'label_cve'),
):
    """Schema-align each partition and enforce deterministic Parquet dtypes."""
    # Normalize incoming column names to canonical names once per partition
    rename_map = {c: raw_to_norm[c] for c in list(df.columns) if c in raw_to_norm and raw_to_norm[c] != c}
    if rename_map:
        try:
            df = df.rename(columns=rename_map)
        except Exception:
            pass

    if len(df) == 0:
        for col in schema_columns:
            if col not in df.columns:
                df[col] = np.nan
        if 'source_file' not in df.columns:
            df['source_file'] = source_name
        return df[schema_columns]

    # Drop columns outside selected schema as early as possible
    keep_now = set(selected_cols) | {'source_file'}
    drop_cols = [c for c in df.columns if c not in keep_now]
    if drop_cols:
        try:
            df = df.drop(columns=drop_cols)
        except Exception:
            pass

    is_cudf = hasattr(df, 'to_pandas')

    # Force string columns to true string dtype to avoid category index overflow
    for col in string_cols:
        if col in df.columns:
            try:
                if is_cudf:
                    df[col] = df[col].fillna('').astype('str')
                else:
                    df[col] = df[col].astype('string').fillna('')
            except Exception:
                pass

    # Optional constrained categoricals (off by default for stability)
    if use_categories:
        blocked = tuple(category_exclude_prefixes)
        for col in string_cols:
            if col not in df.columns:
                continue
            if any(col.startswith(p) for p in blocked):
                continue
            try:
                nunique = int(df[col].nunique(dropna=True))
                ratio = nunique / max(len(df), 1)
                if nunique <= category_max and ratio <= category_ratio:
                    df[col] = df[col].astype('category')
            except Exception:
                pass

    # Force numeric columns and fill medians
    for col in numeric_cols:
        if col in df.columns:
            try:
                df[col] = df[col].astype('float32')
            except Exception:
                pass
            if col in global_medians:
                try:
                    df[col] = df[col].fillna(global_medians[col])
                except Exception:
                    pass

    if 'source_file' not in df.columns:
        df['source_file'] = source_name
    else:
        try:
            if is_cudf:
                df['source_file'] = df['source_file'].astype('str')
            else:
                df['source_file'] = df['source_file'].astype('string')
        except Exception:
            pass

    for col in schema_columns:
        if col not in df.columns:
            if col in string_cols:
                df[col] = ''
            else:
                df[col] = np.nan

    return df[schema_columns]


def process_file_dask(path: Path, schema_columns: list, global_medians: dict) -> dict:
    source_name = path.name
    slug = re.sub(r'[^a-z0-9]+', '_', source_name.lower()).strip('_')
    out_path = RUN_DIR / slug
    tmp_path = RUN_DIR / f'{slug}__tmp'

    try:
        INPROGRESS_PATH.parent.mkdir(parents=True, exist_ok=True)
        INPROGRESS_PATH.write_text(str(path), encoding='utf-8')

        ok, pre = preflight_file(path)
        if not ok:
            return {
                'file': str(path),
                'rows': 0,
                'out': None,
                'error': str(pre),
                'error_type': 'preflight',
                'traceback': None,
            }

        sep = pre['sep']
        enc = pre['enc']
        raw_columns = pre['raw_columns']

        raw_to_norm, normalized_cols = _build_column_maps(raw_columns)
        norm_to_raw = {v: k for k, v in raw_to_norm.items()}

        raw_usecols, dtype_map = _build_usecols_and_dtype(
            path=path,
            sep=sep,
            enc=enc,
            raw_columns=raw_columns,
            raw_to_norm=raw_to_norm,
        )

        if KEEP_COLUMNS:
            selected_cols = [c for c in schema_columns if c == 'source_file' or c in set(KEEP_COLUMNS)]
        else:
            selected_cols = list(schema_columns)

        selected_cols = [c for c in selected_cols if c == 'source_file' or c in set(normalized_cols)]
        if 'source_file' not in selected_cols:
            selected_cols.append('source_file')

        # Determine stable type groups based on sampled dtypes
        string_cols = []
        numeric_cols = []
        for raw_col, raw_dt in dtype_map.items():
            norm_col = raw_to_norm.get(raw_col)
            if norm_col is None or norm_col not in selected_cols:
                continue
            if raw_dt == 'str':
                string_cols.append(norm_col)
            else:
                numeric_cols.append(norm_col)

        # Keep deterministic read kwargs
        read_kwargs = dict(
            sep=sep,
            encoding=enc,
            dtype=dtype_map if dtype_map else None,
            on_bad_lines='skip',
            blocksize=PARTITION_SIZE,
            assume_missing=True,
        )
        if raw_usecols:
            read_kwargs['usecols'] = raw_usecols

        gpu_ok = USE_GPU and dask_cudf is not None
        if gpu_ok:
            try:
                ddf = dask_cudf.read_csv(str(path), **read_kwargs)
                _ = ddf.head(1, compute=True)
            except Exception:
                gpu_ok = False

        if not gpu_ok:
            import dask.dataframe as dd
            ddf = dd.read_csv(str(path), **read_kwargs)

        ddf = ddf.map_partitions(
            clean_partition,
            source_name=source_name,
            global_medians=global_medians,
            schema_columns=schema_columns,
            low_car_max_unique=LOW_CAR_MAX_UNIQUE,
            low_car_max_ratio=LOW_CAR_MAX_RATIO,
            selected_cols=selected_cols,
            string_cols=string_cols,
            numeric_cols=numeric_cols,
            raw_to_norm=raw_to_norm,
            norm_to_raw=norm_to_raw,
            use_gpu=gpu_ok,
            use_categories=False,
        )

        # Build explicit pyarrow schema so all partitions share one physical schema
        import pyarrow as pa

        pa_fields = []
        for col in schema_columns:
            if col == 'source_file' or col in string_cols:
                pa_fields.append(pa.field(col, pa.large_string()))
            else:
                pa_fields.append(pa.field(col, pa.float32()))
        parquet_schema = pa.schema(pa_fields)

        if tmp_path.exists():
            import shutil
            shutil.rmtree(tmp_path, ignore_errors=True)
        tmp_path.mkdir(parents=True, exist_ok=True)

        ddf.to_parquet(
            str(tmp_path),
            write_index=False,
            compression=PARQUET_COMPRESSION,
            write_metadata_file=False,
            overwrite=True,
            schema=parquet_schema,
        )

        rows = int(ddf.map_partitions(len).sum().compute())

        if out_path.exists():
            import shutil
            shutil.rmtree(out_path, ignore_errors=True)
        tmp_path.rename(out_path)

        return {
            'file': str(path),
            'rows': rows,
            'out': str(out_path),
            'error': None,
            'error_type': None,
            'traceback': None,
        }

    except Exception as e:
        return {
            'file': str(path),
            'rows': 0,
            'out': None,
            'error': str(e),
            'error_type': type(e).__name__,
            'traceback': traceback.format_exc(limit=12),
        }

    finally:
        trim_ram()


def process_file_with_retry(path, schema_columns, global_medians, max_retries=2):
    """Retry only transient worker failures; never restart the whole cluster."""
    last_res = None
    for attempt in range(max_retries + 1):
        res = process_file_dask(path, schema_columns, global_medians)
        last_res = res
        if res['error'] is None:
            return res

        if attempt < max_retries and is_transient_worker_error(res['error']):
            trim_ram()
            time.sleep(min(2 ** attempt, 8))
            continue

        return res

    return last_res or {
        'file': str(path),
        'rows': 0,
        'out': None,
        'error': 'unknown error',
        'error_type': 'UnknownError',
        'traceback': None,
    }


def append_failure_log(res: dict):
    if res.get('error') is None:
        return
    FAIL_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(FAIL_LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(res, ensure_ascii=True) + '\n')

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

# -- Cell 8: main pipeline ------------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
if FAIL_LOG_PATH.exists():
    FAIL_LOG_PATH.unlink()

all_files = scan_input_files(DATASET_DIR)
if not all_files:
    raise RuntimeError(f'No CSV/TXT files in {DATASET_DIR}')

if VALIDATE_ONE_FILE_ONLY:
    all_files = all_files[:1]

total_gb = sum(f.stat().st_size for f in all_files) / 1e9
print(f'Files to process: {len(all_files)} | Input size: {total_gb:.2f} GB')

schema_columns, schema_failures = build_global_schema(all_files)
global_medians = sample_medians(all_files, schema_columns)

results = []
total_rows = 0
t0 = pd.Timestamp.now()

for idx, path in enumerate(tqdm(all_files, desc='Files cleaned', unit='file'), start=1):
    print(f'[{idx}/{len(all_files)}] {path.name}')
    res = process_file_with_retry(path, schema_columns, global_medians)
    results.append(res)

    if res.get('error') is None:
        total_rows += int(res['rows'])
    else:
        append_failure_log(res)
        print(f"  FAILED: {res.get('error_type', 'Error')} - {res['error']}")

    trim_ram()

elapsed = (pd.Timestamp.now() - t0).total_seconds()
print(f'Finished in {elapsed/60:.2f} min')

Files to process: 50 | Input size: 95.96 GB


Schema scan:   0%|          | 0/50 [00:00<?, ?file/s]

Sampling medians:   0%|          | 0/50 [00:00<?, ?file/s]

/home/devmrx/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/devmrx/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/devmrx/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/devmrx/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/devmrx/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/devmrx/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_imp

Files cleaned:   0%|          | 0/50 [00:00<?, ?file/s]

[1/50] auth.txt


2026-04-05 22:17:56,500 - distributed.nanny - WARNING - Restarting worker
2026-04-05 22:18:42,270 - distributed.nanny - WARNING - Restarting worker


In [ ]:
# -- Cell 8b: optional single-file validation run -------------------------------
# Use this cell before full processing to validate one representative file.
VALIDATE_ONE_FILE_ONLY = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
all_files = scan_input_files(DATASET_DIR)
if not all_files:
    raise RuntimeError(f'No CSV/TXT files in {DATASET_DIR}')

schema_columns, schema_failures = build_global_schema(all_files[:1])
global_medians = sample_medians(all_files[:1], schema_columns)

res = process_file_with_retry(all_files[0], schema_columns, global_medians)
print('Validation file:', all_files[0])
print('Result:', res)
print('In-progress marker:', INPROGRESS_PATH)

VALIDATE_ONE_FILE_ONLY = False

Schema scan:   0%|          | 0/1 [00:00<?, ?file/s]

Sampling medians:   0%|          | 0/1 [00:00<?, ?file/s]

/home/devmrx/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/devmrx/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/devmrx/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/devmrx/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/devmrx/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/devmrx/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_imp

Validation file: /mnt/e/thesis/dataset/LANL/auth.txt/auth.txt
Result: {'file': '/mnt/e/thesis/dataset/LANL/auth.txt/auth.txt', 'rows': 0, 'out': None, 'error': "An error occurred while calling the read_csv method registered to the pandas backend.\nOriginal Message: Usecols do not match columns, columns expected but not found: ['network', 'c1250', 'logon', 'success', 'anonymous_logon_c586_1', 'ntlm', 'c586', 'anonymous_logon_c586']", 'error_type': 'ValueError', 'traceback': 'Traceback (most recent call last):\n  File "/home/devmrx/miniforge3/lib/python3.13/site-packages/dask/backends.py", line 140, in wrapper\n    return func(*args, **kwargs)\n  File "/home/devmrx/miniforge3/lib/python3.13/site-packages/dask/dataframe/io/csv.py", line 728, in read\n    return read_pandas(\n        reader,\n    ...<10 lines>...\n        **kwargs,\n    )\n  File "/home/devmrx/miniforge3/lib/python3.13/site-packages/dask/dataframe/io/csv.py", line 591, in read_pandas\n    head = reader(BytesIO(b_sample), n

In [ ]:
# -- Cell 8c: debug toolbox (memory + logs + crash file) -----------------------
import subprocess

def show_gpu_memory(interval_sec=2, samples=5):
    cmd = ['nvidia-smi', '--query-gpu=timestamp,memory.used,memory.total,utilization.gpu', '--format=csv,noheader,nounits']
    print('GPU memory snapshots:')
    for _ in range(samples):
        out = subprocess.check_output(cmd, text=True).strip()
        print(out)
        time.sleep(interval_sec)

def show_ram():
    vm = psutil.virtual_memory()
    sm = psutil.swap_memory()
    print(f'RAM used: {(vm.used/1e9):.2f} GB / {(vm.total/1e9):.2f} GB | available: {(vm.available/1e9):.2f} GB')
    print(f'SWAP used: {(sm.used/1e9):.2f} GB / {(sm.total/1e9):.2f} GB')

def tail_worker_logs(n_chars=5000):
    logs = client.get_worker_logs(n=300)
    for worker, txt in logs.items():
        print('\n' + '=' * 80)
        print('Worker:', worker)
        if isinstance(txt, str):
            print(txt[-n_chars:])
        else:
            joined = '\n'.join(str(x) for x in txt)
            print(joined[-n_chars:])

def show_last_in_progress_file():
    if INPROGRESS_PATH.exists():
        print('Last file being processed before failure/restart:')
        print(INPROGRESS_PATH.read_text(encoding='utf-8'))
    else:
        print('No in-progress marker yet.')

print('Debug toolbox ready: show_gpu_memory(), show_ram(), tail_worker_logs(), show_last_in_progress_file()')

Debug toolbox ready: show_gpu_memory(), show_ram(), tail_worker_logs(), show_last_in_progress_file()


In [ ]:
# -- Cell 9: merge partition metadata -------------------------------------------
# Write one _metadata file at RUN_DIR root for efficient downstream reads.
# If schemas differ across fragments, keep only the largest compatible group.

import pyarrow.parquet as pq

succeeded = [r for r in results if not r['error'] and r['out']]

if succeeded:
    meta_list = []
    for r in succeeded:
        try:
            pf = pq.ParquetDataset(r['out'])
            meta_list.extend(pf.fragments)
        except Exception:
            pass

    if meta_list:
        groups = {}
        for frag in meta_list:
            try:
                m = frag.metadata
                if m is None:
                    continue
                k = str(m.schema)
                groups.setdefault(k, []).append(frag)
            except Exception:
                continue

        if groups:
            best_key = max(groups, key=lambda x: len(groups[x]))
            best_group = groups[best_key]

            combined_meta = best_group[0].metadata
            if combined_meta is not None:
                for frag in best_group[1:]:
                    frag_meta = frag.metadata
                    if frag_meta is not None:
                        combined_meta.append_row_groups(frag_meta)

                combined_meta.write_metadata_file(str(RUN_DIR / '_metadata'))

            skipped = len(meta_list) - len(best_group)
            if skipped > 0:
                print(
                    f'Skipped {skipped} fragment(s) with incompatible schemas '
                    f'while building run-level _metadata.'
                )

Skipped 7 fragment(s) with incompatible schemas while building run-level _metadata.


In [ ]:
# -- Cell 10: summary & error report --------------------------------------------
elapsed = (pd.Timestamp.now() - t0).total_seconds()
out_size_gb = sum(f.stat().st_size for f in RUN_DIR.rglob('*.parquet')) / 1e9

succeeded = [r for r in results if r.get('error') is None]
errors = [r for r in results if r.get('error') is not None]

print(f'Total files          : {len(all_files)}')
print(f'Success              : {len(succeeded)}')
print(f'Failed               : {len(errors)}')
print(f'Input size           : {total_gb:.1f} GB')
print(f'Output size          : {out_size_gb:.2f} GB  ->  {RUN_DIR}')
print(f'Total rows written   : {total_rows:,}')
print(f'Elapsed              : {elapsed/60:.1f} min')
print(f'Compression ratio    : {out_size_gb/max(total_gb,1e-9)*100:.1f}%')
print(f'Failure log file     : {FAIL_LOG_PATH}')
print(f'In-progress marker   : {INPROGRESS_PATH}')

if errors:
    print(f'\nFailed files ({len(errors)}):')
    for e in errors[:20]:
        print(f'  {Path(e["file"]).name}: {e.get("error_type", "Error")} - {e["error"]}')
    if len(errors) > 20:
        print(f'  ... and {len(errors) - 20} more (see {FAIL_LOG_PATH})')

    first_tb = next((e.get('traceback') for e in errors if e.get('traceback')), None)
    if first_tb:
        print('\nFirst traceback example:')
        print(first_tb)

if schema_failures:
    print(f'\nSchema scan failures ({len(schema_failures)}):')
    for f, e in schema_failures[:20]:
        print(f'  {Path(f).name}: {e}')
    if len(schema_failures) > 20:
        print(f'  ... and {len(schema_failures) - 20} more')

if len(succeeded) == 0:
    print('\nNo files succeeded. Run Cell 11 (single-file validation) and inspect failure log before full run.')

Total files          : 50
Success              : 8
Failed               : 42
Input size           : 96.0 GB
Output size          : 0.00 GB  ->  /mnt/e/thesis/dataset/dataset_cleaned_parquet/run_outputs
Total rows written   : 171
Elapsed              : 0.8 min
Compression ratio    : 0.0%
Failure log file     : /mnt/e/thesis/dataset_cleaning_failures.jsonl
In-progress marker   : /mnt/e/thesis/dataset_cleaning_in_progress.txt

Failed files (42):
  auth.txt: ValueError - An error occurred while calling the read_csv method registered to the pandas backend.
Original Message: Usecols do not match columns, columns expected but not found: ['network', 'c1250', 'logon', 'success', 'anonymous_logon_c586_1', 'ntlm', 'c586', 'anonymous_logon_c586']
  proc.txt: ValueError - An error occurred while calling the read_csv method registered to the pandas backend.
Original Message: Usecols do not match columns, columns expected but not found: ['start', 'c1', 'c1_dom1', 'p16']
  flows.txt: ValueError - An e

In [ ]:
# -- Cell 11: how to load cleaned data for training -----------------------------
# After cleaning, load from RUN_DIR (not raw CSV) for much faster iterations.

# Option A: load everything (Dask — lazy, out-of-core)
# import dask.dataframe as dd
# ddf = dd.read_parquet(RUN_DIR, engine='pyarrow')
# df  = ddf.compute()

# Option B: load one source only
# import pandas as pd
# df = pd.read_parquet(RUN_DIR / 'your_source_slug')

# Option C: load specific columns only
# import pandas as pd
# df = pd.read_parquet(RUN_DIR, columns=['timestamp', 'source_ip', 'label'])

# Option D: load with cuDF for GPU training
# import cudf
# df = cudf.read_parquet(RUN_DIR / 'your_source_slug')

print('Cleaned Parquet run directory:', RUN_DIR)
print('Sub-folders (one per source file):')
for d in sorted(RUN_DIR.iterdir()):
    if d.is_dir():
        size_mb = sum(f.stat().st_size for f in d.rglob('*.parquet')) / 1e6
        print(f'  {d.name:<50} {size_mb:>8.1f} MB')

Cleaned Parquet run directory: /mnt/e/thesis/dataset/dataset_cleaned_parquet/run_outputs
Sub-folders (one per source file):
  part_00000_0615e778_641c_4399_8d78_f14cb2da50bd_c000_csv__tmp      0.0 MB
  part_00000_0af89d10_df53_44fd_b124_a8a496fd5023_c000_csv__tmp      0.0 MB
  part_00000_0d012414_989a_4b0f_937d_36b02cacf398_c000_csv      0.0 MB
  part_00000_15e3dd03_ea76_429e_a52a_ce96a90517f9_c000_csv__tmp      0.0 MB
  part_00000_20202de3_67ac_4628_90e5_6f7a8153737d_c000_csv      0.0 MB
  part_00000_27b9c76f_8291_49ef_bd5a_c47c18444b18_c000_csv__tmp      0.0 MB
  part_00000_2ac3ee1a_f94a_44bb_9413_dbfa36b751da_c000_csv__tmp      0.0 MB
  part_00000_30aee0c5_aea5_4666_aa1d_36370893753a_c000_1_csv__tmp      0.0 MB
  part_00000_30aee0c5_aea5_4666_aa1d_36370893753a_c000_csv__tmp      0.0 MB
  part_00000_318611a1_7cdc_4dd0_9348_c6368917fd0c_c000_csv__tmp      0.0 MB
  part_00000_4890c258_40e8_4ef8_b345_06d436539b95_c000_csv      0.0 MB
  part_00000_49ed1cc6_3205_4c76_8e28_4da9abf78363_c00

In [ ]:
# ── Cell 12: cleanup ──────────────────────────────────────────────────────────
client.close()
cluster.close()

if USE_GPU and cp is not None:
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()

gc.collect()


654